# Resumen: Consolidación y Enriquecimiento de Datos (Capa Oro)

Este cuaderno implementa la fase final del pipeline de datos (arquitectura Medallion), consolidando la información de la capa *Silver* para su explotación analítica.

* **Descarga e Integración HDFS:** Conecta de forma nativa al clúster Hadoop a través de WebHDFS para descargar las fuentes limpias de clima, consumo (ESIOS) y festivos.
* **Unificación Temporal (Merge):** Fusiona los conjuntos de datos mediante un *inner join* utilizando la variable indexada `fecha` como nexo común.
* **Tipado de Datos Crítico:** Corrige anomalías de formato de texto (`str`) a numérico (`float`) en las métricas térmicas y pluviométricas para posibilitar cálculos algebraicos.
* **Ingeniería de Características (*Feature Engineering*):** Genera nuevas variables de alto valor de negocio: demanda de climatización ($HDD$ y $CDD$), amplitud térmica diaria y acumulados de lluvia diarios.
* **Blindaje y Mapeo Categórico:** Protege el modelo mediante un mapeo determinista de tres estados para el calendario laboral (`Laborable`, `Fin de semana`, `Festivo`) y aplica técnicas de *One-Hot Encoding*.
* **Persistencia en Capa Gold:** Almacena el dataset numérico estructurado en formato Parquet dentro de la nueva ruta optimizada `/datalake/oro/datos_consolidados/` en HDFS.

In [26]:
from hdfs import InsecureClient

import pandas as pd
import numpy as np

In [27]:
%cd ProyectoFinal-IAyBD/oro/data

/home/jovyan/work/ProyectoFinal/oro/data


# Obtencion de datos desde HDFS

In [28]:
client = InsecureClient('http://namenode:9870', user='root')

archivos_a_descargar = [
    ('/datalake/plata/clima/datos_aemet.parquet', 'datos_aemet.parquet'),
    ('/datalake/plata/consumo/datos_esios.parquet', 'datos_esios.parquet'),
    ('/datalake/plata/festivos/datos_festivos.parquet', 'datos_festivos.parquet')
]

for ruta_hdfs, destino_local in archivos_a_descargar:
    print(f"Descargando desde HDFS: {ruta_hdfs}...")
    
    if client.status(ruta_hdfs, strict=False):
        # download(ruta_hdfs, ruta_local, overwrite=True)
        client.download(ruta_hdfs, destino_local, overwrite=True)
        print(f"  - ¡Guardado localmente como '{destino_local}'!\n")
    else:
        print(f"  - ❌ Error: El archivo no existe en la ruta de HDFS especificada.\n")

print("¡Proceso de descarga finalizado!")

Descargando desde HDFS: /datalake/plata/clima/datos_aemet.parquet...
  - ¡Guardado localmente como 'datos_aemet.parquet'!

Descargando desde HDFS: /datalake/plata/consumo/datos_esios.parquet...
  - ¡Guardado localmente como 'datos_esios.parquet'!

Descargando desde HDFS: /datalake/plata/festivos/datos_festivos.parquet...
  - ¡Guardado localmente como 'datos_festivos.parquet'!

¡Proceso de descarga finalizado!


In [29]:

df_aemet = pd.read_parquet('datos_aemet.parquet')
df_esios = pd.read_parquet('datos_esios.parquet')
df_festivos = pd.read_parquet('datos_festivos.parquet')

display(df_aemet.head())
display(df_esios.tail())
display(df_festivos.tail())

,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),fecha
0,Alforja,Tarragona,14.7,5.2,9.9,1.278754,1.000000,0.0,0.0,0.0,0.0,0.0,2026-03-01
1,Reus Aeropuerto,Tarragona,17.3,8.7,13.0,1.414973,1.230449,0.0,0.0,0.0,0.0,0.0,2026-03-01
2,Tarragona,Tarragona,16.1,10.2,13.2,1.255273,0.954243,0.2,0.2,0.0,0.0,0.0,2026-03-01
3,Pontons,Barcelona,12.9,4.6,8.7,1.477121,1.322219,0.0,0.0,0.0,0.0,0.0,2026-03-01
4,Vilafranca del Penedès,Barcelona,18.3,9.2,13.7,1.414973,1.113943,0.0,0.0,0.0,0.0,0.0,2026-03-01


,tipo de energia,valor,porcentaje,fecha
253,Eólica,264705.430,57.91,2026-01-12
254,Eólica,211120.058,49.05,2026-01-11
255,Hidráulica,94637.830,21.99,2026-01-11
256,Hidráulica,76727.058,17.42,2026-01-10
257,Eólica,229100.064,52.01,2026-01-10


,fecha,tipo_dia
1091,2026-12-27,Fin de semana
1092,2026-12-28,Laboral
1093,2026-12-29,Laboral
1094,2026-12-30,Laboral
1095,2026-12-31,Laboral


# Consolidacion de los dataframes

In [30]:
df_consolidado = df_aemet.merge(df_esios, on='fecha', how='inner').merge(df_festivos, on='fecha', how='inner')
df_consolidado.tail()

,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),Precipitación 12-18h (mm),Precipitación 18-24h (mm),fecha,tipo de energia,valor,porcentaje,tipo_dia
103841,Valverde,Santa Cruz de Tenerife,14.7,12.3,13.5,1.623249,1.544068,1.4,0.0,0.0,0.2,1.2,2026-01-31,Hidráulica,147569.863,37.90,Fin de semana
103842,Hierro Aeropuerto,Santa Cruz de Tenerife,21.3,18.2,19.8,1.653213,1.568202,0.0,0.0,0.0,0.0,0.0,2026-01-31,Eólica,180405.313,46.33,Fin de semana
103843,Hierro Aeropuerto,Santa Cruz de Tenerife,21.3,18.2,19.8,1.653213,1.568202,0.0,0.0,0.0,0.0,0.0,2026-01-31,Hidráulica,147569.863,37.90,Fin de semana
103844,"Frontera, Sabinosa",Santa Cruz de Tenerife,21.1,16.6,18.9,1.505150,1.255273,0.0,0.0,0.0,0.0,0.0,2026-01-31,Eólica,180405.313,46.33,Fin de semana
103845,"Frontera, Sabinosa",Santa Cruz de Tenerife,21.1,16.6,18.9,1.505150,1.255273,0.0,0.0,0.0,0.0,0.0,2026-01-31,Hidráulica,147569.863,37.90,Fin de semana


# Creacion de nuevas columnas
### Demanda de climatización estimada (HDD y CDD)
Indicadores económicos estándar en el sector energético llamados Heating Degree Days (Grados Día de Calefacción) y Cooling Degree Days (Grados Día de Refrigeración). Se calculan comparando la temperatura media contra una base de confort (ej. 18ºC)

In [31]:


# HDD (Heating Degree Days): Grados necesarios para calentar si baja de 18ºC
df_consolidado['HDD'] = (18 - pd.to_numeric(df_consolidado['Temperatura media (ºC)'])).clip(lower=0)
# CDD (Cooling Degree Days): Grados necesarios para enfriar si sube de 18ºC
df_consolidado['CDD'] = (pd.to_numeric(df_consolidado['Temperatura media (ºC)']) - 18).clip(lower=0)

df_consolidado[['HDD', 'CDD']].head()

,HDD,CDD
0,8.1,0.0
1,8.1,0.0
2,5.0,0.0
3,5.0,0.0
4,4.8,0.0


# Amplitud Térmica Diaria
Diferencia entre la temperatura máxima y la mínima del día

In [32]:
df_consolidado['Amplitud_Termica'] = df_consolidado['Temperatura máxima (ºC)'] - df_consolidado['Temperatura mínima (ºC)']
df_consolidado['Amplitud_Termica'] .head()

0    9.5
1    9.5
2    8.6
3    8.6
4    5.9
Name: Amplitud_Termica, dtype: float64

# Precipitación Total Diaria Calculada
Sumamos de forma robusta los 4 tramos de 6 horas del CSV

In [33]:
df_consolidado['Precipitacion_Total_Calculada'] = (
    df_consolidado['Precipitación 00-06h (mm)'] + 
    df_consolidado['Precipitación 06-12h (mm)'] + 
    df_consolidado['Precipitación 12-18h (mm)'] + 
    df_consolidado['Precipitación 18-24h (mm)']
)

df_consolidado['Precipitacion_Total_Calculada'].head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.2
Name: Precipitacion_Total_Calculada, dtype: float64

Mostrar un vistazo del resultado para comprobar

In [34]:
df_consolidado[['fecha', 'Estación', 'HDD', 'CDD', 'Amplitud_Termica', 'Precipitacion_Total_Calculada']].head()

,fecha,Estación,HDD,CDD,Amplitud_Termica,Precipitacion_Total_Calculada
0,2026-03-01,Alforja,8.1,0.0,9.5,0.0
1,2026-03-01,Alforja,8.1,0.0,9.5,0.0
2,2026-03-01,Reus Aeropuerto,5.0,0.0,8.6,0.0
3,2026-03-01,Reus Aeropuerto,5.0,0.0,8.6,0.0
4,2026-03-01,Tarragona,4.8,0.0,5.9,0.2


# Pasar datos categoricos a numericos

- Aplicamos el mapeo a tipo de dias: Fin de semana = 0, laborable = 1 y Festivo = 2
- Aplicamos One-Hot Encoding a la columna de tipo de energia

In [35]:
mapa_tipo_dia = {
    'Fin de semana': 0,
    'Laborable': 1,
    'Festivo': 2
}

df_consolidado['tipo_dia_encoded'] = df_consolidado['tipo_dia'].map(mapa_tipo_dia)
df_consolidado = pd.get_dummies(df_consolidado, columns=['tipo de energia'], drop_first=True, dtype=int)

df_consolidado.head()

,Estación,Provincia,Temperatura máxima (ºC),Temperatura mínima (ºC),Temperatura media (ºC),Racha (km/h),Velocidad máxima (km/h),Precipitación 00-24h (mm),Precipitación 00-06h (mm),Precipitación 06-12h (mm),...,fecha,valor,porcentaje,tipo_dia,HDD,CDD,Amplitud_Termica,Precipitacion_Total_Calculada,tipo_dia_encoded,tipo de energia_Hidráulica
0,Alforja,Tarragona,14.7,5.2,9.9,1.278754,1.000000,0.0,0.0,0.0,...,2026-03-01,130518.326,25.84,Fin de semana,8.1,0.0,9.5,0.0,0.0,1
1,Alforja,Tarragona,14.7,5.2,9.9,1.278754,1.000000,0.0,0.0,0.0,...,2026-03-01,228914.087,45.32,Fin de semana,8.1,0.0,9.5,0.0,0.0,0
2,Reus Aeropuerto,Tarragona,17.3,8.7,13.0,1.414973,1.230449,0.0,0.0,0.0,...,2026-03-01,130518.326,25.84,Fin de semana,5.0,0.0,8.6,0.0,0.0,1
3,Reus Aeropuerto,Tarragona,17.3,8.7,13.0,1.414973,1.230449,0.0,0.0,0.0,...,2026-03-01,228914.087,45.32,Fin de semana,5.0,0.0,8.6,0.0,0.0,0
4,Tarragona,Tarragona,16.1,10.2,13.2,1.255273,0.954243,0.2,0.2,0.0,...,2026-03-01,130518.326,25.84,Fin de semana,4.8,0.0,5.9,0.2,0.0,1


In [36]:
df_consolidado.to_parquet("datos_consolidados.parquet", index=False)

# Subir archivo a HDSF

In [37]:
client = InsecureClient('http://namenode:9870', user='root')

ruta_hdfs_oro = '/datalake/oro/datos/datos_consolidados.parquet'

if client.status(ruta_hdfs_oro, strict=False):
    print(f"  - Detectado archivo antiguo en {ruta_hdfs_oro}. Borrando...")
    client.delete(ruta_hdfs_oro)
    print("  - Archivo antiguo eliminado.")

client.upload(ruta_hdfs_oro, 'datos_consolidados.parquet')

print(f"\n¡Proceso completado! Archivo subido con éxito a la capa Oro: {ruta_hdfs_oro}")

  - Detectado archivo antiguo en /datalake/oro/datos/datos_consolidados.parquet. Borrando...
  - Archivo antiguo eliminado.

¡Proceso completado! Archivo subido con éxito a la capa Oro: /datalake/oro/datos/datos_consolidados.parquet


# Llamar a script del dashboard

In [38]:
from pathlib import Path
import runpy


def localizar_script() -> Path:
    cwd = Path.cwd().resolve()
    candidatos = [
        cwd / "oro" / "dashboard_df_consolidado.py",
        cwd / "dashboard_df_consolidado.py",
        cwd.parent / "oro" / "dashboard_df_consolidado.py",
        cwd.parent / "dashboard_df_consolidado.py",
    ]
    for candidato in candidatos:
        if candidato.exists():
            return candidato

    for candidato in cwd.rglob("dashboard_df_consolidado.py"):
        return candidato

    raise FileNotFoundError("No se encontró dashboard_df_consolidado.py en el workspace del kernel.")


script_path = localizar_script()
runpy.run_path(str(script_path), run_name="__main__")


Dashboard generado en: /home/jovyan/work/ProyectoFinal/oro/data/dashboard_df_consolidado.html


{'__name__': '__main__',
 '__doc__': None,
 '__package__': '',
 '__loader__': None,
 '__spec__': None,
 '__file__': '/home/jovyan/work/ProyectoFinal/oro/dashboard_df_consolidado.py',
 '__cached__': None,
 '__builtins__': {'__name__': 'builtins',
  '__doc__': "Built-in functions, types, exceptions, and other objects.\n\nThis module provides direct access to all 'built-in'\nidentifiers of Python; for example, builtins.len is\nthe full name for the built-in function len().\n\nThis module is not normally accessed explicitly by most\napplications, but can be useful in modules that provide\nobjects with the same name as a built-in value, but in\nwhich the built-in of that name is also needed.",
  '__package__': '',
  '__loader__': _frozen_importlib.BuiltinImporter,
  '__spec__': ModuleSpec(name='builtins', loader=<class '_frozen_importlib.BuiltinImporter'>, origin='built-in'),
  '__build_class__': <function __build_class__>,
  '__import__': <function __import__(name, globals=None, locals=Non